In [1]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from smmargins import Margins

rng = np.random.default_rng(7)
N = 5_000
df = pd.DataFrame({
    "age":    rng.normal(45, 12, N).clip(18, 90),
    "income": rng.lognormal(10.5, 0.4, N),
    "educ":   rng.choice(["hs", "college", "grad"], N, p=[0.4, 0.4, 0.2]),
    "female": rng.integers(0, 2, N),
})
eta = (-4.0 + 0.05 * df["age"] + 0.00001 * df["income"]
       + 0.8 * (df["educ"] == "college") + 1.4 * (df["educ"] == "grad")
       + 0.3 * df["female"] - 0.0004 * df["age"] * df["female"])
df["voted"] = (rng.uniform(0, 1, N) < 1 / (1 + np.exp(-eta))).astype(int)

fit = smf.logit("voted ~ age + income + C(educ) + female + age:female", data=df).fit(disp=False)
M = Margins(fit)

In [2]:
M_robust = Margins(fit, cov_type="HC3")
M_robust.dydx("age").summary()

,dy/dx,std err,z,P>|z|,[95% Conf.,Interval]
dage,0.010118,0.000499,20.275469,2.117617e-91,0.00914,0.011096


In [3]:
M.dydx("age", vce="simulation", n_sims=2000, sim_seed=42).summary()

,dy/dx,std err,z,P>|z|,[95% Conf.,Interval]
dage,0.010118,0.000497,20.374085,2.839719e-92,0.009108,0.011088


In [4]:
M.dydx("female", vce="simulation", n_sims=2000, sim_seed=42).summary()

,contrast,std err,z,P>|z|,[95% Conf.,Interval]
female: 1 vs 0,0.062486,0.012995,4.808567,0.000002,0.036856,0.088527


In [5]:
M.dydx("age", vce="bootstrap", n_boot=500, boot_seed=42).summary()

,dy/dx,std err,z,P>|z|,[95% Conf.,Interval]
dage,0.010118,0.000505,20.023489,3.437809e-89,0.00919,0.011156


In [6]:
pred_grid = M.predict(atexog={"age": [25, 45, 65]})
pred_grid.summary()

,prediction,std err,z,P>|z|,[95% Conf.,Interval]
age=25,0.176056,0.009420,18.690339,5.933527e-78,0.157594,0.194518
age=45,0.354894,0.006769,52.430979,0.000000e+00,0.341627,0.368160
age=65,0.583282,0.013865,42.069884,0.000000e+00,0.556107,0.610456


In [7]:
M.predict(
    atexog={"age": [25, 45, 65]},
    vce="simulation",
    n_sims=4000,
    ci_method="bonferroni"
).summary()

,prediction,std err,z,P>|z|,[95% Conf.,Interval]
age=25,0.176056,0.009374,18.781311,1.073957e-78,0.153615,0.198497
age=45,0.354894,0.006852,51.794139,0.000000e+00,0.338490,0.371297
age=65,0.583282,0.013784,42.314722,0.000000e+00,0.550282,0.616281


In [8]:
M.predict(
    atexog={"age": [25, 45, 65]},
    vce="simulation",
    n_sims=4000,
    ci_method="sidak"
).summary()

,prediction,std err,z,P>|z|,[95% Conf.,Interval]
age=25,0.176056,0.009312,18.906793,1.002733e-79,0.153822,0.198290
age=45,0.354894,0.006756,52.529234,0.000000e+00,0.338762,0.371026
age=65,0.583282,0.013680,42.639080,0.000000e+00,0.550618,0.615945


In [9]:
M.predict(
    atexog={"age": [25, 45, 65]},
    vce="simulation",
    n_sims=4000,
    ci_method="sup-t"
).summary()

,prediction,std err,z,P>|z|,[95% Conf.,Interval]
age=25,0.176056,0.009387,18.754517,1.778273e-78,0.153973,0.198139
age=45,0.354894,0.006804,52.155944,0.000000e+00,0.338887,0.370901
age=65,0.583282,0.013953,41.803332,0.000000e+00,0.550458,0.616105
